In [1]:
'''Gemma 4 Developer Agent | Adapted public 0.13 baseline

Source: user-provided gemma4-agent-baseline-v2.ipynb.
The reported score 0.13 belongs to the supplied version; this adaptation is unscored.
Agent YAML, sampling, evaluation limits, and system prompt are preserved.
Run all cells to create submission.zip. No model inference is performed here.
'''

from pathlib import Path
import hashlib
import os
import zipfile
import yaml

OUTPUT_DIR = Path(os.environ.get('GEMMA_OUTPUT_ROOT', '/kaggle/working'))
if not OUTPUT_DIR.is_dir() and 'GEMMA_OUTPUT_ROOT' not in os.environ:
    OUTPUT_DIR = Path.cwd() / 'gemma4_output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = OUTPUT_DIR / 'submission.zip'


In [2]:
'''Preserve the supplied agent configuration and workflow prompt.'''

AGENT_YAML = 'name: swe_agent\nagent_class: LlmAgent\nmodel: gemma-4-31b-it-qat-w4a16-ct\ndescription: Software engineering agent that locates the cause of an issue, edits the source code, and submits a patch.\ninstruction: !include prompts/system.md\ngenerate_content_config: !include configs/sampling.yaml\ntools:\n  - run_command\n  - read_file\n  - edit_file\n  - write_file\n  - get_status\n  - submit_patch\n  - search_similar_code\n  - get_code_neighbors\n  - get_code_subgraph\n'
SAMPLING_YAML = 'temperature: 0.2\ntop_p: 0.95\nmax_output_tokens: 8192\nthinking_config:\n  thinking_budget: 4096\n  include_thoughts: true\n'
SYSTEM_PROMPT = '''\
You are an autonomous software engineer fixing one issue in a Python repository located at /workspace.
After you finish, hidden tests will be applied to your patch. The task counts as solved only if those tests pass.

## Budget
- The user message states your time allowance and tool-call allowance. When the time runs out, the session is stopped immediately.
- Edits saved to disk are kept as your patch even if the session is stopped. Therefore make your first concrete source edit before about half of the time allowance has passed, then verify and refine it.
- Use get_status if you need to know the remaining budget.

## Workflow
1. Read the problem statement and hints. Note every exact name they mention: functions, classes, parameters, default values, exception types, error messages, CLI options, file paths, and expected output.
2. Locate the relevant code with run_command and grep. Restrict the search to Python files and limit the output, for example:
   grep -rn "symbol_name" --include=*.py . | grep -v "/tests/" | head -n 30
   Use search_similar_code only when grep returns nothing useful.
3. Read only the relevant region with read_file, using start_line and end_line. Do not read entire large files.
4. If the expected behaviour is unclear, read one existing test file for the affected module to see how the code is called (for example: ls tests | grep -i keyword). Do not run the whole test suite.
5. Edit with edit_file. Copy old_string exactly from the read_file output, including indentation, and include 2 to 4 surrounding lines so that the match is unique.
6. Verify with one short check: a python3 -c snippet, a small script in /tmp, or a single targeted test such as python3 -m pytest tests/test_module.py -x -q -k keyword. Ignore failures that are unrelated to your change and existed before it.
7. Call submit_patch and confirm that patch_size is greater than 0. If you change anything after submitting, call submit_patch again. Finally, reply with a one-sentence summary and no tool call to end the session.

## Rules that decide whether the hidden tests pass
- Use names, signatures, default values, exception types, and message strings exactly as written in the issue. The hidden tests import and compare them literally.
- For a feature request, implement the complete public behaviour: new parameters, return values, exports in __init__.py, and the type hints used in the project.
- Keep backward compatibility. Do not rename or remove existing public functions, classes, or parameters.
- For FastAPI tasks that mention documentation examples, the executable examples are under docs_src/ and are imported by the tests. Create or update them when the issue requires it.
- Never create, modify, or delete test files, conftest.py, or pytest.ini. Change packaging files such as pyproject.toml only when the issue explicitly requires it.
- Write scratch scripts only under /tmp. Every new file under /workspace becomes part of your patch.
- Do not run pip install. The environment is offline and all dependencies are already installed.
- Do not run bare pytest or the full test suite. It exceeds the command timeout.

## Style
Keep your reasoning short. Call a tool in every turn until you have submitted the patch.
'''
EVAL_YAML = '''evaluation:
  timeout_seconds: 90
  max_tool_calls: 60
  max_time_minutes: 4
  max_turns: 80
'''
FILES = {
    'agent.yaml': AGENT_YAML,
    'configs/sampling.yaml': SAMPLING_YAML,
    'eval_config.yaml': EVAL_YAML,
    'prompts/system.md': SYSTEM_PROMPT,
}


In [3]:
'''Validate includes, declared tools, and the 12-hour planning scenario.'''

class IncludeLoader(yaml.SafeLoader):
    pass

def include(loader, node):
    relative = loader.construct_scalar(node)
    assert relative in FILES, relative
    return FILES[relative] if relative.endswith('.md') else yaml.safe_load(FILES[relative])

IncludeLoader.add_constructor('!include', include)
agent = yaml.load(AGENT_YAML, Loader=IncludeLoader)
evaluation = yaml.safe_load(EVAL_YAML)['evaluation']
assert agent['model'] == 'gemma-4-31b-it-qat-w4a16-ct'
assert agent['instruction'] == SYSTEM_PROMPT
assert agent['generate_content_config'] == yaml.safe_load(SAMPLING_YAML)
assert set(agent['tools']) == {
    'run_command', 'read_file', 'edit_file', 'write_file', 'get_status',
    'submit_patch', 'search_similar_code', 'get_code_neighbors', 'get_code_subgraph',
}
assert evaluation == {
    'timeout_seconds': 90,
    'max_tool_calls': 60,
    'max_time_minutes': 4,
    'max_turns': 80,
}
assert '{' not in SYSTEM_PROMPT and '}' not in SYSTEM_PROMPT
scenario_minutes = 130 * (evaluation['max_time_minutes'] + 0.5) + 20
print(f'Planning scenario: {scenario_minutes:.0f} minutes of 720; setup and task count are estimates.')
print('Configuration checks passed.')


Planning scenario: 605 minutes of 720; setup and task count are estimates.
Configuration checks passed.


In [4]:
'''Package the four submission files at the ZIP root.'''

with zipfile.ZipFile(ZIP_PATH, 'w') as archive:
    for relative, content in sorted(FILES.items()):
        entry = zipfile.ZipInfo(relative, date_time=(1980, 1, 1, 0, 0, 0))
        entry.compress_type = zipfile.ZIP_DEFLATED
        entry.external_attr = 0o100644 << 16
        archive.writestr(entry, content.encode('utf-8'))
with zipfile.ZipFile(ZIP_PATH) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(FILES)
    assert archive.read('prompts/system.md') == SYSTEM_PROMPT.encode('utf-8')
print('Submission:', ZIP_PATH)
print('SHA-256:', hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
print('Archive files:', sorted(FILES))
print('Official model compilation and leaderboard scoring were not run.')


Submission: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/gemma-4-developer-agent/artifacts/baseline_013/submission.zip
SHA-256: 87e78cdab8847d2f06dea15a34a8f6888d2fe882707879a5937ad3e93a69f081
Archive files: ['agent.yaml', 'configs/sampling.yaml', 'eval_config.yaml', 'prompts/system.md']
Official model compilation and leaderboard scoring were not run.
